# 4. Análisis de Estructura de Equipos y Jerarquía

**Objetivo:** Visualizar la estructura de los departamentos, identificando managers y la cantidad de empleados que reportan a cada uno para entender la carga de trabajo y la estructura de los equipos.

In [6]:
import pandas as pd
import psycopg2
import getpass
import matplotlib.pyplot as plt
import seaborn as sns

### Conexión a la Base de Datos
Por favor, introduce las credenciales de la base de datos a continuación.

In [7]:
db_name = input("Database Name: ")
db_user = input("Database User: ")
db_pass = getpass.getpass("Database Password: ")
db_host = "postgres"
db_port = "5432"

try:
    conn = psycopg2.connect(
        dbname=db_name,
        user=db_user,
        password=db_pass,
        host=db_host,
        port=db_port
    )
    print("Conexión a la base de datos exitosa.")
except psycopg2.OperationalError as e:
    print(f"Error en la conexión: {e}")

Database Name:  employees
Database User:  admin
Database Password:  ········


Conexión a la base de datos exitosa.


### (Debug) Verificar datos de managers y empleados actuales

In [8]:
debug_query_managers = "SELECT COUNT(*) FROM dept_manager WHERE to_date = '9999-12-31';"
debug_query_dept_emp = "SELECT COUNT(*) FROM dept_emp WHERE to_date = '9999-12-31';"

try:
    managers_count = pd.read_sql_query(debug_query_managers, conn).iloc[0,0]
    dept_emp_count = pd.read_sql_query(debug_query_dept_emp, conn).iloc[0,0]
    print(f"Registros en 'dept_manager' con to_date='9999-12-31': {managers_count}")
    print(f"Registros en 'dept_emp' con to_date='9999-12-31': {dept_emp_count}")
except Exception as e:
    print(f"Error durante la depuración: {e}")

Registros en 'dept_manager' con to_date='9999-12-31': 0
Registros en 'dept_emp' con to_date='9999-12-31': 0


/tmp/ipykernel_3039/471750328.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  managers_count = pd.read_sql_query(debug_query_managers, conn).iloc[0,0]
/tmp/ipykernel_3039/471750328.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dept_emp_count = pd.read_sql_query(debug_query_dept_emp, conn).iloc[0,0]


### Consulta SQL para obtener la estructura de equipos

In [9]:
query = """
WITH manager_info AS (
    SELECT 
        dm.dept_no,
        d.dept_name,
        dm.emp_no AS manager_id,
        CONCAT(e.first_name, ' ', e.last_name) AS manager_name
    FROM 
        dept_manager dm
    JOIN 
        employees e ON dm.emp_no = e.emp_no
    JOIN
        departments d ON dm.dept_no = d.dept_no
    WHERE 
        dm.to_date = '9999-12-31'
),
employee_counts AS (
    SELECT 
        de.dept_no,
        COUNT(de.emp_no) - 1 AS num_employees -- Restamos 1 para no contar al manager
    FROM 
        dept_emp de
    WHERE 
        de.to_date = '9999-12-31'
    GROUP BY
        de.dept_no
)
SELECT 
    mi.dept_name,
    mi.manager_name,
    ec.num_employees
FROM 
    manager_info mi
JOIN 
    employee_counts ec ON mi.dept_no = ec.dept_no
ORDER BY
    ec.num_employees DESC;
"""

try:
    df = pd.read_sql_query(query, conn)
    print("DataFrame de estructura de equipos:")
    display(df.head(10))
except Exception as e:
    print(f"Error al ejecutar la consulta: {e}")

DataFrame de estructura de equipos:


/tmp/ipykernel_3039/4160812657.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,dept_name,manager_name,num_employees


### Visualización de la Carga de Trabajo de los Managers

In [10]:
plt.style.use('seaborn-v0_8-whitegrid')
if not df.empty:
    plt.figure(figsize=(14, 8))
    ax = sns.barplot(data=df, x='num_employees', y='dept_name', palette='cubehelix', orient='h')
    
    plt.title('Número de Empleados por Departamento (excl. Manager)', fontsize=16)
    plt.xlabel('Número de Empleados', fontsize=12)
    plt.ylabel('Departamento', fontsize=12)
    plt.tight_layout()
    
    # Añadir etiquetas con el nombre del manager
    for index, row in df.iterrows():
        ax.text(row.num_employees + 500, index, f"Mgr: {row.manager_name}", color='black', ha="left", va="center")
    
    plt.show()
else:
    print("El DataFrame está vacío. No se puede generar el gráfico de carga de trabajo.")

El DataFrame está vacío. No se puede generar el gráfico de carga de trabajo.


### Distribución de la Carga de Managers

In [11]:
if not df.empty:
    plt.figure(figsize=(10, 6))
    sns.histplot(df['num_employees'], bins=10, kde=True, color='purple')
    plt.title('Distribución del Tamaño de los Equipos', fontsize=16)
    plt.xlabel('Número de Empleados por Equipo', fontsize=12)
    plt.ylabel('Frecuencia (Nº de Departamentos)', fontsize=12)
    plt.axvline(df['num_employees'].mean(), color='red', linestyle='--', label=f"Media: {df['num_employees'].mean():.2f}")
    plt.legend()
    plt.show()
else:
    print("El DataFrame está vacío. No se puede generar el gráfico de distribución de carga.")

# Cerrar la conexión
if 'conn' in locals() and conn is not None:
    conn.close()
    print("Conexión cerrada.")

El DataFrame está vacío. No se puede generar el gráfico de distribución de carga.
Conexión cerrada.
